# Module 6: AI-Assisted Analytics Pattern

**Unit C · Week 6 + Extension** · Track 2 — Visualization, Analytic Hubs & Responsible AI

The track's signature pattern: share only the data **schema** with an LLM, run the code it returns **locally**, and pass on only an **aggregate summary**. `call_llm()` below is a stand-in for a real API call so this runs with no API key — swap in your provider's client and the rest of the pattern is unchanged.

## Learning objectives

- **Intermediate** _Apply_ — Build a working, multi-filter dashboard with at least one chart and one map wired together.
- **Advanced** _Analyze / Create, via the Extension_ — Integrate an AI-assisted analytics pattern (schema-only code generation, local execution, summary-only narration) with data-minimization safeguards designed in from the start.
- **Advanced** _Analyze / Create_ — Port a dashboard component's logic between Python and R without loss of correctness.

## Setup

This notebook reads `../../data/processed/track2_dataset.csv`, built by
`data/make_sample_data.py`. It is **synthetic** data shaped like the real
NISR/HDX handoff described in the course site's Chapter 3 — swap in a real
extract by re-pointing the path below once you have one. See
`data/README.md` for where to get real data and exactly what to rename.


In [1]:
import pandas as pd

DATA_PATH = "../../data/processed/track2_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
print(f"{len(df):,} rows · {df['district'].nunique()} districts · "
      f"{df['date'].min().date()} to {df['date'].max().date()}")
df.head()

240 rows · 10 districts · 2023-01-01 to 2024-12-01


,date,province,district,district_pcode,indicator,value,feature_1,feature_2,outcome
0,2023-01-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,39.37,82.01,19.22,1
1,2023-02-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,45.24,49.99,17.62,1
2,2023-03-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,41.16,46.66,19.93,0
3,2023-04-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.91,49.17,20.35,0
4,2023-05-01,Kigali City,Gasabo,SIM-GAS,sample_wellbeing_index,42.05,51.04,20.35,0


In [2]:
def call_llm(prompt: str) -> str:
    """Stand-in for a real LLM call. In the exercise, call your provider's
    API here. Whatever it returns is executed locally in Step 2 — the
    model only ever sees the schema from Step 1, never a data row."""
    return "result = df.groupby('district')['value'].mean().sort_values(ascending=False).head(5)"


# Step 1: share only the schema, never raw rows
schema = {c: str(t) for c, t in df.dtypes.items()}
prompt = f"Data frame schema (columns and types only): {schema}\nWrite pandas code that returns the top 5 districts by mean 'value'. Return code only, no data."
generated_code = call_llm(prompt)
print("LLM returned:\n ", generated_code)

# Step 2: run it locally, against the real data
local_vars = {"df": df}
exec(generated_code, {}, local_vars)
result = local_vars["result"]

# Step 3: only the aggregate result is passed on
narration_prompt = f"Summarize this result in one paragraph: {result.to_dict()}"
print("\nWhat leaves the machine (aggregate only):\n ", result.to_dict())

LLM returned:
  result = df.groupby('district')['value'].mean().sort_values(ascending=False).head(5)

What leaves the machine (aggregate only):
  {'Gicumbi': 65.41166666666668, 'Kicukiro': 62.11083333333334, 'Nyarugenge': 60.346666666666664, 'Musanze': 59.83416666666667, 'Nyagatare': 53.532916666666665}


## Your turn

Extend the Module 5 prototype into a full dashboard (at least 2 filters, 1 chart, 1 map); add one AI-assisted feature built strictly to the schema-only / local-execution / summary-only pattern below; port one component to the other language.

**Formative assessment.** Live demo + code review: a 5-minute live dashboard demo graded against a reactivity/design rubric.

## Responsible AI callback

The schema-only / local-execution / summary-only pattern is itself a data-minimization safeguard, named explicitly here so it is a practiced habit before Module 9 formalizes it as a governance requirement.